# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [2]:
import numpy as np

query = "latency vs throughput"
# Creamos 5 documentos de prueba
docs = [f"doc{i}: texto del documento {i}" for i in range(5)]

# Scores simulados (ya están alineados por índice con los docs)
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60]) 
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3]) 

# ==========================================
# 1. PRIMERA ETAPA: Recuperación (Bi-encoder)
# ==========================================
# np.argsort ordena de menor a mayor. Tomamos los 4 últimos (los más altos) y los invertimos con [::-1]
top_4_indices_bi = np.argsort(bi_scores)[-4:][::-1]

print("--- ETAPA 1: Top 4 (Bi-encoder) ---")
for idx in top_4_indices_bi:
    print(f"{docs[idx]} -> Score: {bi_scores[idx]:.2f}")

# ==========================================
# 2. SEGUNDA ETAPA: Re-ranking (Cross-encoder)
# ==========================================
# Extraemos los cross_scores SOLO de los 4 documentos ganadores
scores_filtrados = cross_scores[top_4_indices_bi]

# Ordenamos esos 4 internamente según su nuevo cross-score
indices_relativos_cross = np.argsort(scores_filtrados)[::-1]

print("\n--- ETAPA 2: Orden Final (Cross-encoder) ---")
for i, rel_idx in enumerate(indices_relativos_cross):
    # Recuperamos el índice original mapeando a través de top_4_indices_bi
    idx_original = top_4_indices_bi[rel_idx]
    print(f"Top {i+1}: {docs[idx_original]} -> Cross-Score: {cross_scores[idx_original]:.2f} (Bi-Score original: {bi_scores[idx_original]:.2f})")

--- ETAPA 1: Top 4 (Bi-encoder) ---
doc1: texto del documento 1 -> Score: 0.81
doc3: texto del documento 3 -> Score: 0.78
doc0: texto del documento 0 -> Score: 0.72
doc4: texto del documento 4 -> Score: 0.60

--- ETAPA 2: Orden Final (Cross-encoder) ---
Top 1: doc1: texto del documento 1 -> Cross-Score: 0.90 (Bi-Score original: 0.81)
Top 2: doc3: texto del documento 3 -> Cross-Score: 0.85 (Bi-Score original: 0.78)
Top 3: doc4: texto del documento 4 -> Cross-Score: 0.30 (Bi-Score original: 0.60)
Top 4: doc0: texto del documento 0 -> Cross-Score: 0.10 (Bi-Score original: 0.72)


## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [3]:
# TODO: MMR pseudocode as comments or stub function
def seleccion_mmr(docs_candidatos, query, k_resultados, lambda_param=0.5):
    seleccionados = []
    
    while len(seleccionados) < k_resultados and docs_candidatos:
        # La magia del MMR en 1 línea: maximizar (Relevancia) - (Redundancia)
        mejor_doc = max(docs_candidatos, key=lambda d: lambda_param * similitud(d, query) - (1 - lambda_param) * max([similitud(d, s) for s in seleccionados], default=0))
        
        seleccionados.append(mejor_doc)
        docs_candidatos.remove(mejor_doc)
        
    return seleccionados

## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa del Pipeline | Coste relativo (Escala aprox.) | Nivel de Latencia |
| :--- | :--- | :--- |
| **1. Embedding único de query** (Bi-encoder) | **1x** | 🟢 **Muy Bajo** (Milisegundos) |
| **2. $k$ llamadas Cross-encoder** (Re-ranking) | **10x — 50x** (Depende de $k$) | 🟡 **Medio** (Décimas de segundo) |
| **3. Generación LLM (200 tokens)** | **1000x+** | 🔴 **Muy Alto** (Segundos) |